In [14]:
!pip install  docling  langchain-milvus
!pip install langchain openai langchain_openai pymilvus


In [1]:
import logging
import time
from pathlib import Path

logging.basicConfig(level=logging.INFO)
_log = logging.getLogger(__name__)

In [38]:
import openai
from dotenv import load_dotenv
import os
load_dotenv()  # Load biến môi trường từ file .env
from openai import OpenAI


openai_api_key = os.getenv("OPENAI_API_KEY")
  # Kiểm tra xem biến môi trường đã được tải chưa
openai_client = OpenAI(api_key=openai_api_key)



In [108]:
from docling_core.types.doc import ImageRefMode
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, PictureDescriptionApiOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import granite_picture_description

def openai_vlm_options():
    return PictureDescriptionApiOptions(
        url="https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {openai_api_key}"},
        params=dict(
            model="gpt-4o-mini",  # hoặc gpt-4o
            max_tokens=200,
        ),
        # prompt chung để mô tả ảnh
        prompt="Diễn dải hình ảnh một cách chi tiết và chính xác bằng tiếng Việt.",
        timeout=60,
    )

input_pdf_path = "../../../data_sample/viettailieukhoahoc_cut.pdf"  # Thay bằng đường dẫn PDF của bạn
# output_dir = Path("output")                                           # Thư mục lưu Markdown (ảnh nằm trong artifact)
# output_dir.mkdir(parents=True, exist_ok=True)

IMAGE_RESOLUTION_SCALE = 2.0  # Độ phân giải ảnh xuất ra

# ------------------------------
# Cấu hình pipeline của Docling để tạo ảnh trong artifact
# ------------------------------
pipeline_options = PdfPipelineOptions()
pipeline_options.images_scale = IMAGE_RESOLUTION_SCALE

# pipeline_options.generate_picture_images = True  # Tạo figures trong artifact
pipeline_options.do_ocr = True
pipeline_options.do_table_structure = True
# pipeline_options.table_structure_options.do_cell_matching = True
pipeline_options.enable_remote_services=True
pipeline_options.ocr_options.lang = ["es"]
pipeline_options.do_picture_description = True
pipeline_options.picture_description_options = (
    openai_vlm_options()
)
# pipeline_options.picture_description_options.prompt = (
#     "Describe the image in three sentences. Be consise and accurate."
# )




In [109]:
doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

start_time = time.time()
conv_res = doc_converter.convert(input_pdf_path)
doc_filename = conv_res.input.file.stem

# ------------------------------
# Xuất Markdown, ảnh tự động nằm trong artifact
# ------------------------------


end_time = time.time() - start_time
_log.info(f"PDF converted to Markdown in {end_time:.2f} seconds")

INFO:docling.datamodel.document:detected formats: [<InputFormat.PDF: 'pdf'>]
INFO:docling.document_converter:Going to convert document batch...
INFO:docling.document_converter:Initializing pipeline for StandardPdfPipeline with options hash c5ae9db392956289316fee7c12e72ed3
INFO:docling.utils.accelerator_utils:Accelerator device: 'cpu'
INFO:docling.utils.accelerator_utils:Accelerator device: 'cpu'
INFO:docling.utils.accelerator_utils:Accelerator device: 'cpu'
INFO:docling.pipeline.base_pipeline:Processing document viettailieukhoahoc_cut.pdf
c:\Users\Admin\anaconda3\envs\vector_db\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\Admin\anaconda3\envs\vector_db\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
 

In [110]:
from collections import Counter

item_counter = Counter()

for element, _ in conv_res.document.iterate_items():
    item_counter[type(element).__name__] += 1

print("Các loại item trong tài liệu và số lượng:")
for name, count in item_counter.most_common():
    print(f"- {name}: {count}")


Các loại item trong tài liệu và số lượng:
- TextItem: 104
- ListItem: 19
- SectionHeaderItem: 18
- TableItem: 14
- PictureItem: 5


In [111]:
from docling_core.types.doc import PictureItem, TableItem, SectionHeaderItem, ListItem

first_picture = None

for element, _ in conv_res.document.iterate_items():
    if isinstance(element, TableItem):
        first_picture = element
        break  # dừng lại ngay khi tìm được ảnh đầu tiên

if first_picture:
    print("Kiểu:", type(first_picture).__name__)
    print("Tất cả thuộc tính:")
    for k, v in first_picture.__dict__.items():
        if not k.startswith("_"):
            print(f"  {k}: {v}")
else:
    print("Không tìm thấy PictureItem nào trong tài liệu.")


Kiểu: TableItem
Tất cả thuộc tính:
  self_ref: #/tables/0
  parent: cref='#/body'
  children: [RefItem(cref='#/texts/24')]
  content_layer: ContentLayer.BODY
  label: table
  prov: [ProvenanceItem(page_no=3, bbox=BoundingBox(l=119.3786849975586, t=728.4894561767578, r=475.29095458984375, b=674.5360260009766, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 0))]
  captions: [RefItem(cref='#/texts/24')]
  references: []
  footnotes: []
  image: None
  data: table_cells=[TableCell(bbox=BoundingBox(l=126.5, t=120.06799999999998, r=178.697, b=128.14, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), row_span=1, col_span=1, start_row_offset_idx=0, end_row_offset_idx=1, start_col_offset_idx=0, end_col_offset_idx=1, text='Nhiệt độ ( \uf0b0 C)', column_header=False, row_header=False, row_section=False, fillable=False), TableCell(bbox=BoundingBox(l=199.25, t=119.58799999999997, r=262.907, b=127.65999999999997, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), row_span=1, col_span=1

In [112]:
from docling.chunking import HybridChunker

chunker = HybridChunker()
chunk_iter = chunker.chunk(dl_doc=conv_res.document)

chunks = list(chunk_iter)        # chuyển generator → list
print(len(chunks))    

Token indices sequence length is longer than the specified maximum sequence length for this model (642 > 512). Running this sequence through the model will result in indexing errors


89


In [115]:
# for c in chunks:
#     print(c.meta.model_dump())  # hoặc print(c) để xem nội dung chunk

print(chunks[11].model_dump())

{'text': '-50, Sinh trưởng sau 48 giờ (mm) = 0. -40, Sinh trưởng sau 48 giờ (mm) = 0. -30, Sinh trưởng sau 48 giờ (mm) = 0. -20, Sinh trưởng sau 48 giờ (mm) = 0. -10, Sinh trưởng sau 48 giờ (mm) = 0. 0, Sinh trưởng sau 48 giờ (mm) = 0. 10, Sinh trưởng sau 48 giờ (mm) = 0. 20, Sinh trưởng sau 48 giờ (mm) = 7. 30, Sinh trưởng sau 48 giờ (mm) = 8. 40, Sinh trưởng sau 48 giờ (mm) = 1. 50, Sinh trưởng sau 48 giờ (mm) = 0. 60, Sinh trưởng sau 48 giờ (mm) = 0. 70, Sinh trưởng sau 48 giờ (mm) = 0. 80, Sinh trưởng sau 48 giờ', 'meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/tables/1', 'parent': {'cref': '#/body'}, 'children': [{'cref': '#/texts/27'}], 'content_layer': <ContentLayer.BODY: 'body'>, 'label': <DocItemLabel.TABLE: 'table'>, 'prov': [{'page_no': 3, 'bbox': {'l': 193.23764038085938, 't': 516.1500854492188, 'r': 401.55731201171875, 'b': 217.38128662109375, 'coord_origin': <CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>}, 'char

In [35]:
import json

# Lưu chunk và kích thước metadata
chunk_sizes = []
for c in chunks:
    meta_dict = c.meta.model_dump()  # dict
    meta_json = json.dumps(meta_dict, default=str)  # convert sang string
    size = len(meta_json)  # độ dài string = “khối lượng”
    chunk_sizes.append((c, size))

# Sắp xếp theo size giảm dần
chunk_sizes.sort(key=lambda x: x[1], reverse=True)

# In ra top 5 chunk metadata nhiều nhất
for c, size in chunk_sizes[:5]:
    print(f"Size: {size}, Text preview: {c.text[:100]}...")


Size: 2541, Text preview: Toàn bộ bảng 8.11 và đoạn viết: 'Nhóm lợn thương phẩm (Com) … (P > 0,05) (Bảng 8.11).' là phần viết ...
Size: 2307, Text preview: - (1) Mở đầu bằng cách tóm tắt bối cảnh, giả thuyết, mục tiêu và phát hiện chính của nghiên cứu;
- (...
Size: 2149, Text preview: Vịt T14.± SE = 62,00
Ghi chú: Các giá trị trung bình trên cùng một hàng nếu có chữ cái khác nhau sự ...
Size: 2147, Text preview: 3. **Số ngày điều trị (màu vàng)**: Cột này có giá trị 14, biểu thị tổng số ngày điều trị cho các gi...
Size: 1884, Text preview: 3. **Các đường biểu diễn**:
   - **Đường màu xanh lá cây (2010-2011)**: Xu hướng tăng mạnh từ đầu nă...


In [127]:
from dotenv import load_dotenv
import os
load_dotenv()  # Load biến môi trường từ file .env

MILVUS_HOST = os.getenv("MILVUS_HOST")
MILVUS_PORT = os.getenv("MILVUS_PORT")
MILVUS_USER = os.getenv("MILVUS_USER")
MILVUS_PASSWORD = os.getenv("MILVUS_PASSWORD")

In [128]:
from pymilvus import connections

# Kết nối bằng endpoint và token
conn = connections.connect(
    alias="default",
    host=MILVUS_HOST,  # thay bằng endpoint của bạn
    port=MILVUS_PORT,          
    user=MILVUS_USER,         # thay bằng user từ token
    password=MILVUS_PASSWORD, # thay bằng password từ token
    secure=True          # HTTPS cần secure=True
)
  

# Kiểm tra kết nối
print(connections.list_connections())

[('default', <pymilvus.client.grpc_handler.GrpcHandler object at 0x00000204B71CF690>), ('https://in03-90dd51f2ca86e25.serverless.aws-eu-central-1.cloud.zilliz.com:443-db_90dd51f2ca86e25', <pymilvus.client.grpc_handler.GrpcHandler object at 0x00000204AFC4A7D0>), ('async-https://in03-90dd51f2ca86e25.serverless.aws-eu-central-1.cloud.zilliz.com:443-db_90dd51f2ca86e25', <pymilvus.client.async_grpc_handler.AsyncGrpcHandler object at 0x00000204B04ED2D0>)]


In [129]:
from pymilvus import Collection, FieldSchema, CollectionSchema, DataType, Function, FunctionType, connections

from pymilvus import MilvusClient

milvus_client = MilvusClient(
    uri=f"https://{MILVUS_HOST}:{MILVUS_PORT}",
    user=MILVUS_USER,
    password=MILVUS_PASSWORD,
    secure=True
)



In [ ]:
schema = milvus_client.create_schema(auto_id=False)

# Các trường trong schema
schema.add_field(
    field_name="pk",
    datatype=DataType.VARCHAR,
    max_length=1000,
    is_primary=True,
    description="primary key"
)

schema.add_field(
    field_name="page_content",
    datatype=DataType.VARCHAR,
    max_length=65535,
    enable_analyzer=True,
    description="raw text content"
)

schema.add_field(
    field_name="embedding",
    datatype=DataType.FLOAT_VECTOR,
    dim=1536,
    description="dense embedding vector"
)

schema.add_field(
    field_name="sparse",
    datatype=DataType.SPARSE_FLOAT_VECTOR,
    description="sparse BM25 vector"
)

schema.add_field(
    field_name="metadata",
    datatype=DataType.JSON,
    description="metadata field"
)

# 3️⃣ Thêm BM25 function cho sparse vector
bm25_function = Function(
    name="text_bm25_emb",
    input_field_names=["page_content"],
    output_field_names=["sparse"],
    function_type=FunctionType.BM25
)
schema.add_function(bm25_function)

# 4️⃣ Chuẩn bị index cho từng trường
index_params = milvus_client.prepare_index_params()

# Dense vector
index_params.add_index(
    field_name="embedding",
    index_name="embedding_index",
    index_type="IVF_FLAT",
    metric_type="IP",
    params={"nlist": 128}
)

# Sparse BM25 vector
index_params.add_index(
    field_name="sparse",
    index_name="sparse_index",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="BM25",
    params={"inverted_index_algo": "DAAT_MAXSCORE"}  # or "DAAT_WAND", "TAAT_NAIVE"
)

# 5️⃣ Tạo collection với schema và index
milvus_client.create_collection(
    collection_name="my_collection",
    schema=schema,
    index_params=index_params
)

print("Collection 'my_collection' created successfully!")

In [94]:

milvus_client.load_collection("my_collection")

res = milvus_client.get_load_state(
    collection_name="my_collection"
)

print(res)

{'state': <LoadState: Loaded>}


In [41]:
from pprint import pprint 
pprint(chunks[0].meta)

DocMeta(schema_name='docling_core.transforms.chunker.DocMeta', version='1.0.0', doc_items=[DocItem(self_ref='#/texts/2', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, label=<DocItemLabel.TEXT: 'text'>, prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=85.104, t=684.3760439453125, r=513.032, b=596.0440439453125, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 430))])], headings=['KẾT QUẢ VÀ THẢO LUẬN'], captions=None, origin=DocumentOrigin(mimetype='application/pdf', binary_hash=16291828457832834346, filename='viettailieukhoahoc_cut.pdf', uri=None))


In [95]:
def emb_text(text):
    return (
        openai_client.embeddings.create(input=text, model="text-embedding-3-small")
        .data[0]
        .embedding
    )

In [96]:
import json
from uuid import uuid4
from tqdm import tqdm

collection_name = "my_collection"
data = []

for chunk in tqdm(chunks, desc="Processing chunks"):
    embedding = emb_text(chunk.text)
    data.append({
        "pk": str(uuid4()),                       # UUID cho primary key
        "embedding": embedding,
        "page_content": chunk.text,
        "metadata": json.dumps(chunk.meta.model_dump())  # serialize DocMeta sang JSON
    })

milvus_client.insert(collection_name=collection_name, data=data)


Processing chunks: 100%|██████████| 90/90 [00:45<00:00,  1.96it/s]


{'insert_count': 90, 'ids': ['5b2c7bf9-28fb-48a7-9f0b-0cf175e28c44', 'c3cad236-e9bb-4797-8d6a-64e61d25de44', '32279f15-615e-4643-8df1-9a6ad76eaf1c', '58cbb904-f087-47ad-8974-08faba1a4b13', '750c0324-cb34-436a-ae9e-ef349f8fd46b', '342a93f8-4b2f-4da6-ba98-386895eb0b2b', '66c36cc3-7223-4456-a9ac-f20bdfff0a60', '21cdfcda-601b-42ec-9df7-1bfa1e822d30', '02069a85-fc92-4be6-9617-596b275fcb62', '506130df-08bc-4397-8ac5-55b46825efb0', '3352dd09-9d8e-4358-b3d4-d549911d6360', 'f49c169a-9a08-42e2-a6cd-23c259d0db70', 'fe0cfa61-d38b-4ce5-987f-fae3647bb271', 'c51db5d8-a320-444b-9c97-e4823bb5143c', '1fa10e1c-4234-4b97-928f-62784817eb9e', '6ba5a2ea-56a2-406d-a640-42055cdbcf99', 'a9237d9a-a7b3-4508-98e6-932cbeca963a', 'e6055590-f744-4401-b358-a92ff0590698', '6c861341-6256-4769-b6c1-60c914a6298d', '8297daa8-79e3-4ad5-9d92-853cd19262c4', 'ccbaa2ea-f382-47ca-ba19-1292da9d4fb5', '11686578-7354-4793-b4a3-ff5c3cb87b0d', '8b689e65-cdd5-4ce6-97ce-ac62be12062b', 'b0119e1e-435f-4b7a-a0d5-c2fce0a87490', 'bd1995e0-0

**Search theo Spare vector(BM25)**

In [125]:
search_params = {
    'params': {'drop_ratio_search': 0.2},
}

results = milvus_client.search(
    collection_name='my_collection', 
    data=['Ảnh hưởng của môi trường hiếu khí đối với sinh trưởng của nấm Streptomyces coelicolor '],
    anns_field='sparse',
    output_fields=['page_content'], # Fields to return in search results; sparse field cannot be output
    limit=3,
    search_params=search_params
)

print("=== Search Results ===")
for i, hits in enumerate(results, start=1):   # mỗi query trả về 1 list hits
    print(f"\nQuery {i}:")
    for j, hit in enumerate(hits, start=1):
        print(f"  Result {j}:")
        
        print(f"    PageContent: {hit['entity']['page_content']}")

=== Search Results ===

Query 1:
  Result 1:
    PageContent: Không nên tạo lập các  bảng  có  quá  ít  số  liệu,  hoặc  có  quá  nhiều  số  liệu  trùng nhau. Sau đây là một vài ví dụ minh họa:
Bảng 8.1. Ảnh hưởng của môi trường hiếu khí đối với sinh trưởng của nấm Streptomyces coelicolor

Nhiệt độ (  C), 1 = Số thực nghiệm. Nhiệt độ (  C), 2 = Môi trường hiếu khí. Nhiệt độ (  C), 3 = Độ sinh trưởng (Klett). 24, 1 = 5. 24, 2 = +. 24, 3 = 78. 24, 1 = 5. 24, 2 = -. 24, 3 = 0
Nguồn: Day, 1998
  Result 2:
    PageContent: Có thể nhận thấy: bảng trên có rất ít dữ liệu, một vài dữ liệu lại trùng lặp. Trong trường hợp này, thay cho việc lập bảng số liệu, chỉ cần viết một đoạn văn bản như sau: 'Môi trường hiếu khí là cần thiết cho sinh trưởng của nấm Streptomices coelicolor . Với nhiệt độ trong phòng (24 0 C), nấm không sinh trưởng trong môi trường yếm khí, nhưng sinh trưởng mạnh (78 độ Klett) trong môi trường hiếu khí'. Đoạn văn bản này vừa ngắn gọn vừa nêu được toàn bộ các dữ liệu thu đượ

**search theo similarity**

In [126]:
results = milvus_client.search(
    collection_name="my_collection",
    data=[emb_text("Ảnh hưởng của môi trường hiếu khí đối với sinh trưởng của nấm Streptomyces coelicolor")],                # phải là vector chứ không phải string
    anns_field="embedding",          # Tìm trên dense embedding
    output_fields=["page_content", "metadata"],
    limit=3,
    search_params=search_params
)

print("=== Search Results ===")
for i, hits in enumerate(results, start=1):   # mỗi query trả về 1 list hits
    print(f"\nQuery {i}:")
    for j, hit in enumerate(hits, start=1):
        print(f"  Result {j}:")
        
        print(f"    PageContent: {hit['entity']['page_content']}")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


=== Search Results ===

Query 1:
  Result 1:
    PageContent: Có thể nhận thấy: bảng trên có rất ít dữ liệu, một vài dữ liệu lại trùng lặp. Trong trường hợp này, thay cho việc lập bảng số liệu, chỉ cần viết một đoạn văn bản như sau: 'Môi trường hiếu khí là cần thiết cho sinh trưởng của nấm Streptomices coelicolor . Với nhiệt độ trong phòng (24 0 C), nấm không sinh trưởng trong môi trường yếm khí, nhưng sinh trưởng mạnh (78 độ Klett) trong môi trường hiếu khí'. Đoạn văn bản này vừa ngắn gọn vừa nêu được toàn bộ các dữ liệu thu được.

Bảng 8.2. Ảnh hưởng của nhiệt độ tới sự nảy mầm của hạt sồi (Quercus)
  Result 2:
    PageContent: Không nên tạo lập các  bảng  có  quá  ít  số  liệu,  hoặc  có  quá  nhiều  số  liệu  trùng nhau. Sau đây là một vài ví dụ minh họa:
Bảng 8.1. Ảnh hưởng của môi trường hiếu khí đối với sinh trưởng của nấm Streptomyces coelicolor

Nhiệt độ (  C), 1 = Số thực nghiệm. Nhiệt độ (  C), 2 = Môi trường hiếu khí. Nhiệt độ (  C), 3 = Độ sinh trưởng (Klett). 24, 1 = 5